# Compile ph and summary data from runs with only PM or PFA sweep

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
path_main = Path("../../simulations")

path_data = path_main / "20251113_summary"

# Path to save output csv
path_csv = path_main / "20251113_summary_refined"
if not path_csv.exists():
    path_csv.mkdir(parents=True, exist_ok=True)

In [3]:
def load_summary(search_type, cr, br, pm, pfa):
    return pd.read_csv(
        path_data / f"{search_type}_cr{cr}_br{br}_pm{pm:.0e}_pfa{pfa:.0e}_summary.csv",
        index_col=0
    )

In [4]:
def load_ph(search_type, cr, br, pm, pfa):
    return xr.open_dataset(
        path_data / f"{search_type}_cr{cr}_br{br}_pm{pm:.0e}_pfa{pfa:.0e}_ph.nc"
    )

In [5]:
def get_ping_cross_p_max_th(ds, p_max_th=0.95):    
    ping_cross_all = []
    for run in ds["run"].values:
        ping_cross_th = ds["p_max"].sel(run=run).dropna(dim="ping").values > p_max_th
        if ping_cross_th.sum() > 0:
            ping_cross = np.argwhere(ping_cross_th).min()
        else:
            ping_cross = 999
        ping_cross_all.append(int(ping_cross))
    return np.array(ping_cross_all)

In [6]:
def get_ping_p_max_diff_th(ds, pmax_diff_threshold=1e-5):
    criteria = {
        "pmax_repeat_N": 3,
        "max_ping_num": 500,
        "pmax_diff_threshold": pmax_diff_threshold,
    }    
    p_all = []
    for run in ds["run"].values:
        p_max = ds["p_max"].sel(run=run).dropna(dim="ping")
        for p in range(len(p_max)):
            if p - criteria["pmax_repeat_N"] < 0:
                continue
            p_max_diff = np.diff(p_max[p-criteria["pmax_repeat_N"]+1:p+1])
            if len(p_max_diff) > 1 and np.all(abs(p_max_diff) < criteria["pmax_diff_threshold"]):
                p_all.append(p)
                break
    return np.array(p_all)

In [7]:
def refine_num_pings_pfa_fixed(cr, br, pm_all, pfa_fixed):
    for idx, pm in enumerate(pm_all):
        print("------------------------------------------------")
        print(f"cr={cr}, br={br}, pm={pm}, pfa={pfa_fixed}")

        # Load summary and ph details
        df_infotaxis = load_summary("infotaxis", cr, br, pm, pfa=pfa_fixed)
        df_MAP = load_summary("MAP", cr, br, pm, pfa=pfa_fixed)

        # Get num_pings when p_max crosses threshold
        ds_infotaxis = load_ph("infotaxis", cr, br, pm, pfa=pfa_fixed)
        ds_MAP = load_ph("MAP", cr, br, pm, pfa=pfa_fixed)
        x1 = get_ping_cross_p_max_th(ds_infotaxis)
        y1 = get_ping_cross_p_max_th(ds_MAP)
        x2 = get_ping_p_max_diff_th(ds_infotaxis, pmax_diff_threshold=1e-5)
        y2 = get_ping_p_max_diff_th(ds_MAP, pmax_diff_threshold=1e-5)

        # Substitute num_pings in the runs that did not produce pmax > threshold
        # with pmax convergence num_pings
        idx_no_cross_x = x1==999
        x = x1.copy()
        x[idx_no_cross_x] = x2[idx_no_cross_x]
        idx_no_cross_y = y1==999
        y = y1.copy()
        y[idx_no_cross_y] = y2[idx_no_cross_y]

        # Store refined number of pings into the summary df
        df_infotaxis["num_pings_cross_pmax_th"] = x1
        df_infotaxis["num_pings_pmax_diff"] = x2
        df_infotaxis["num_pings_2conditions"] = x
        df_MAP["num_pings_cross_pmax_th"] = y1
        df_MAP["num_pings_pmax_diff"] = y2
        df_MAP["num_pings_2conditions"] = y

        fname_postfix = f"cr{cr}_br{br}_pm{pm:.0e}_pfa{pfa_fixed:.0e}_summary.csv"
        df_infotaxis.to_csv(path_csv / f"infotaxis_{fname_postfix}")
        df_MAP.to_csv(path_csv / f"MAP_{fname_postfix}")

        print("save refined info to:")
        print(f" - infotaxis_{fname_postfix}")
        print(f" - MAP_{fname_postfix}")

In [8]:
cr_all = [5, 8, 10]
br_all = [1, 2]
pm_all = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
pfa_fixed = 0

In [9]:
for cr in cr_all:
    for br in br_all:
            refine_num_pings_pfa_fixed(cr, br, pm_all, pfa_fixed)

------------------------------------------------
cr=5, br=1, pm=0.001, pfa=0
save refined info to:
 - infotaxis_cr5_br1_pm1e-03_pfa0e+00_summary.csv
 - MAP_cr5_br1_pm1e-03_pfa0e+00_summary.csv
------------------------------------------------
cr=5, br=1, pm=0.002, pfa=0
save refined info to:
 - infotaxis_cr5_br1_pm2e-03_pfa0e+00_summary.csv
 - MAP_cr5_br1_pm2e-03_pfa0e+00_summary.csv
------------------------------------------------
cr=5, br=1, pm=0.005, pfa=0
save refined info to:
 - infotaxis_cr5_br1_pm5e-03_pfa0e+00_summary.csv
 - MAP_cr5_br1_pm5e-03_pfa0e+00_summary.csv
------------------------------------------------
cr=5, br=1, pm=0.01, pfa=0
save refined info to:
 - infotaxis_cr5_br1_pm1e-02_pfa0e+00_summary.csv
 - MAP_cr5_br1_pm1e-02_pfa0e+00_summary.csv
------------------------------------------------
cr=5, br=1, pm=0.02, pfa=0
save refined info to:
 - infotaxis_cr5_br1_pm2e-02_pfa0e+00_summary.csv
 - MAP_cr5_br1_pm2e-02_pfa0e+00_summary.csv
-------------------------------------

In [10]:
def refine_num_pings_pm_fixed(cr, br, pm_fixed, pfa_all):
    for idx, pfa in enumerate(pfa_all):
        print("------------------------------------------------")
        print(f"cr={cr}, br={br}, pfa={pfa}, pm={pm_fixed}")

        # Load summary and ph details
        df_infotaxis = load_summary("infotaxis", cr, br, pm_fixed, pfa)
        df_MAP = load_summary("MAP", cr, br, pm_fixed, pfa)

        # Get num_pings when p_max crosses threshold
        ds_infotaxis = load_ph("infotaxis", cr, br, pm_fixed, pfa)
        ds_MAP = load_ph("MAP", cr, br, pm_fixed, pfa=pfa)
        x1 = get_ping_cross_p_max_th(ds_infotaxis)
        y1 = get_ping_cross_p_max_th(ds_MAP)
        x2 = get_ping_p_max_diff_th(ds_infotaxis, pmax_diff_threshold=1e-5)
        y2 = get_ping_p_max_diff_th(ds_MAP, pmax_diff_threshold=1e-5)

        # Substitute num_pings in the runs that did not produce pmax > threshold
        # with pmax convergence num_pings
        idx_no_cross_x = x1==999
        x = x1.copy()
        x[idx_no_cross_x] = x2[idx_no_cross_x]
        idx_no_cross_y = y1==999
        y = y1.copy()
        y[idx_no_cross_y] = y2[idx_no_cross_y]

        # Store refined number of pings into the summary df
        df_infotaxis["num_pings_cross_pmax_th"] = x1
        df_infotaxis["num_pings_pmax_diff"] = x2
        df_infotaxis["num_pings_2conditions"] = x
        df_MAP["num_pings_cross_pmax_th"] = y1
        df_MAP["num_pings_pmax_diff"] = y2
        df_MAP["num_pings_2conditions"] = y

        fname_postfix = f"cr{cr}_br{br}_pm{pm_fixed:.0e}_pfa{pfa:.0e}_summary.csv"
        df_infotaxis.to_csv(path_csv / f"infotaxis_{fname_postfix}")
        df_MAP.to_csv(path_csv / f"MAP_{fname_postfix}")

        print("save refined info to:")
        print(f" - infotaxis_{fname_postfix}")
        print(f" - MAP_{fname_postfix}")

In [11]:
cr_all = [5, 8, 10]
br_all = [1, 2]
pm_fixed = 0
pfa_all = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]

In [12]:
for cr in cr_all:
    for br in br_all:
            refine_num_pings_pm_fixed(cr, br, pm_fixed, pfa_all)

------------------------------------------------
cr=5, br=1, pfa=0.001, pm=0
save refined info to:
 - infotaxis_cr5_br1_pm0e+00_pfa1e-03_summary.csv
 - MAP_cr5_br1_pm0e+00_pfa1e-03_summary.csv
------------------------------------------------
cr=5, br=1, pfa=0.002, pm=0
save refined info to:
 - infotaxis_cr5_br1_pm0e+00_pfa2e-03_summary.csv
 - MAP_cr5_br1_pm0e+00_pfa2e-03_summary.csv
------------------------------------------------
cr=5, br=1, pfa=0.005, pm=0
save refined info to:
 - infotaxis_cr5_br1_pm0e+00_pfa5e-03_summary.csv
 - MAP_cr5_br1_pm0e+00_pfa5e-03_summary.csv
------------------------------------------------
cr=5, br=1, pfa=0.01, pm=0
save refined info to:
 - infotaxis_cr5_br1_pm0e+00_pfa1e-02_summary.csv
 - MAP_cr5_br1_pm0e+00_pfa1e-02_summary.csv
------------------------------------------------
cr=5, br=1, pfa=0.02, pm=0
save refined info to:
 - infotaxis_cr5_br1_pm0e+00_pfa2e-02_summary.csv
 - MAP_cr5_br1_pm0e+00_pfa2e-02_summary.csv
-------------------------------------